In [ ]:
# cell 1
import warnings

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import (
    precision_score,
    recall_score,
    roc_auc_score,
    accuracy_score,
    f1_score
)
from sklearn.exceptions import ConvergenceWarning
from imblearn.over_sampling import RandomOverSampler


In [ ]:
# cell 2
# Settings
DATA_DIR = r"..\Data\model_data"
RESULTS_DIR = r"..\Results\Model_selection"

SEED = 10
N_SPLITS = 5

OUTCOME_PREFIXES = [
    "mask",
    "protective",
    "social_avoidance"
]

warnings.filterwarnings("ignore", category=ConvergenceWarning)


In [ ]:
# cell 3
# Cross-validation setup
cv = StratifiedShuffleSplit(
    n_splits=N_SPLITS,
    test_size=1 / N_SPLITS,
    random_state=SEED
)

In [ ]:
# cell 4
# Helper functions
def load_training_data(prefix):
    X_train = pd.read_csv(f"{DATA_DIR}\\{prefix}_X_train.csv")
    y_train = pd.read_csv(f"{DATA_DIR}\\{prefix}_y_train.csv").values.ravel()

    return X_train, y_train


def prepare_fold_data(X_train, X_val):
    X_train = X_train.copy()
    X_val = X_val.copy()

    bool_cols_train = X_train.select_dtypes(include=["bool"]).columns
    bool_cols_val = X_val.select_dtypes(include=["bool"]).columns

    if len(bool_cols_train) > 0:
        X_train[bool_cols_train] = X_train[bool_cols_train].astype(int)

    if len(bool_cols_val) > 0:
        X_val[bool_cols_val] = X_val[bool_cols_val].astype(int)

    X_train, X_val = X_train.align(
        X_val,
        join="left",
        axis=1,
        fill_value=0
    )

    return X_train, X_val


def summarize_cv_scores(cv_scores):
    summary_df = pd.DataFrame({
        "fold": cv_scores["fold"],
        "precision": cv_scores["precision"],
        "recall": cv_scores["recall"],
        "roc_auc": cv_scores["roc_auc"],
        "accuracy": cv_scores["accuracy"],
        "f1": cv_scores["f1"]
    })

    mean_row = pd.DataFrame([{
        "fold": "mean",
        "precision": summary_df["precision"].mean(),
        "recall": summary_df["recall"].mean(),
        "roc_auc": summary_df["roc_auc"].mean(),
        "accuracy": summary_df["accuracy"].mean(),
        "f1": summary_df["f1"].mean()
    }])

    se_row = pd.DataFrame([{
        "fold": "se",
        "precision": summary_df["precision"].std(ddof=1) / np.sqrt(N_SPLITS),
        "recall": summary_df["recall"].std(ddof=1) / np.sqrt(N_SPLITS),
        "roc_auc": summary_df["roc_auc"].std(ddof=1) / np.sqrt(N_SPLITS),
        "accuracy": summary_df["accuracy"].std(ddof=1) / np.sqrt(N_SPLITS),
        "f1": summary_df["f1"].std(ddof=1) / np.sqrt(N_SPLITS)
    }])

    summary_df = pd.concat(
        [summary_df, mean_row, se_row],
        ignore_index=True
    )

    return summary_df


def get_mean_se(summary_df, metric):
    mean_value = summary_df.loc[
        summary_df["fold"] == "mean",
        metric
    ].values[0]

    se_value = summary_df.loc[
        summary_df["fold"] == "se",
        metric
    ].values[0]

    return mean_value, se_value


def print_mean_se(summary_df, prefix):
    print("\n" + "=" * 60)
    print(f"Logistic Regression results for: {prefix}")

    for metric in ["precision", "recall", "roc_auc", "accuracy", "f1"]:
        mean_value, se_value = get_mean_se(summary_df, metric)
        print(f"{metric}: {mean_value:.3f} ± {se_value:.3f}")

In [ ]:
# cell 5
# Model evaluation
def evaluate_logistic_regression(prefix, upsample=True):
    X, y = load_training_data(prefix)

    cv_scores = {
        "fold": [],
        "precision": [],
        "recall": [],
        "roc_auc": [],
        "accuracy": [],
        "f1": []
    }

    for fold_idx, (train_idx, val_idx) in enumerate(cv.split(X, y), start=1):

        X_train_fold = X.iloc[train_idx].copy()
        y_train_fold = y[train_idx].copy()

        X_val_fold = X.iloc[val_idx].copy()
        y_val_fold = y[val_idx].copy()

        if upsample:
            upsampler = RandomOverSampler(random_state=SEED)
            X_train_fold, y_train_fold = upsampler.fit_resample(
                X_train_fold,
                y_train_fold
            )

            if not isinstance(X_train_fold, pd.DataFrame):
                X_train_fold = pd.DataFrame(
                    X_train_fold,
                    columns=X.columns
                )

        X_train_prepared, X_val_prepared = prepare_fold_data(
            X_train_fold,
            X_val_fold
        )

        model = LogisticRegression(
            max_iter=5000,
            random_state=SEED
        )

        model.fit(X_train_prepared, y_train_fold)

        y_pred = model.predict(X_val_prepared)
        y_prob = model.predict_proba(X_val_prepared)[:, 1]

        cv_scores["fold"].append(fold_idx)
        cv_scores["precision"].append(
            precision_score(y_val_fold, y_pred, zero_division=0)
        )
        cv_scores["recall"].append(
            recall_score(y_val_fold, y_pred, zero_division=0)
        )
        cv_scores["roc_auc"].append(
            roc_auc_score(y_val_fold, y_prob)
        )
        cv_scores["accuracy"].append(
            accuracy_score(y_val_fold, y_pred)
        )
        cv_scores["f1"].append(
            f1_score(y_val_fold, y_pred, zero_division=0)
        )

    return cv_scores

In [ ]:
# cell 6
# Cross-validate logistic regression
def cross_validate_logistic_regression(prefix, upsample=True):
    print("\n" + "=" * 60)
    print(f"Cross-validating Logistic Regression for: {prefix}")
    print(f"Upsampling: {upsample}")

    cv_scores = evaluate_logistic_regression(
        prefix=prefix,
        upsample=upsample
    )

    summary_df = summarize_cv_scores(cv_scores)

    summary_df.to_csv(
        f"{RESULTS_DIR}\\lr_{prefix}_cv_summary.csv",
        index=False
    )

    print_mean_se(summary_df, prefix)

    return summary_df


In [ ]:
# cell 7
# Run logistic regression CV
lr_summaries = {}

for prefix in OUTCOME_PREFIXES:
    summary_df = cross_validate_logistic_regression(
        prefix=prefix,
        upsample=True
    )

    lr_summaries[prefix] = summary_df


Cross-validating Logistic Regression for: mask
Upsampling: True

Logistic Regression results for: mask
precision: 0.799 ± 0.004
recall: 0.770 ± 0.003
roc_auc: 0.843 ± 0.002
accuracy: 0.771 ± 0.002
f1: 0.784 ± 0.002

Cross-validating Logistic Regression for: protective
Upsampling: True

Logistic Regression results for: protective
precision: 0.825 ± 0.002
recall: 0.704 ± 0.004
roc_auc: 0.786 ± 0.003
accuracy: 0.710 ± 0.003
f1: 0.760 ± 0.003

Cross-validating Logistic Regression for: social_avoidance
Upsampling: True

Logistic Regression results for: social_avoidance
precision: 0.764 ± 0.002
recall: 0.680 ± 0.003
roc_auc: 0.758 ± 0.003
accuracy: 0.687 ± 0.002
f1: 0.720 ± 0.002


In [ ]:
# cell 8
# Build comparison table
rows = []

for prefix, summary_df in lr_summaries.items():
    row = {
        "model": "Logistic Regression",
        "outcome": prefix,
        "upsample": True
    }

    for metric in ["precision", "recall", "roc_auc", "accuracy", "f1"]:
        mean_value, se_value = get_mean_se(summary_df, metric)

        row[metric] = mean_value
        row[f"{metric}_se"] = se_value
        row[f"{metric}_mean_se"] = f"{mean_value:.3f} ± {se_value:.3f}"

    rows.append(row)

lr_comparison_df = pd.DataFrame(rows)

lr_comparison_df.to_csv(
    f"{RESULTS_DIR}\\lr_cv_comparison.csv",
    index=False
)

lr_comparison_df

,model,outcome,upsample,precision,precision_se,precision_mean_se,recall,recall_se,recall_mean_se,roc_auc,roc_auc_se,roc_auc_mean_se,accuracy,accuracy_se,accuracy_mean_se,f1,f1_se,f1_mean_se
0,Logistic Regression,mask,True,0.798507,0.003961,0.799 ± 0.004,0.769870,0.003220,0.770 ± 0.003,0.842666,0.002429,0.843 ± 0.002,0.770944,0.002279,0.771 ± 0.002,0.783878,0.001841,0.784 ± 0.002
1,Logistic Regression,protective,True,0.825233,0.002324,0.825 ± 0.002,0.703872,0.003853,0.704 ± 0.004,0.785983,0.003292,0.786 ± 0.003,0.709966,0.003467,0.710 ± 0.003,0.759729,0.003168,0.760 ± 0.003
2,Logistic Regression,social_avoidance,True,0.764306,0.002466,0.764 ± 0.002,0.680327,0.003454,0.680 ± 0.003,0.758187,0.002834,0.758 ± 0.003,0.687325,0.001586,0.687 ± 0.002,0.719838,0.001694,0.720 ± 0.002
